# Pig Posture Recognition - V8a Training (DINOv2-L + Pseudo-Labels)

**Ansatz:** DINOv2 Large + Pseudo-Labels aus V7-Ensemble (B_dinov2_strong, 0.862 Kaggle).

**Warum nur DINOv2-L?** EVA-02 brachte im V7-Ensemble nur +0.002 Kaggle-Gewinn gegenueber
DINOv2-only (0.862 vs 0.860) - unterhalb der Rausch-Schwelle. ConvNeXt V2-L war zu schwach
(0.76 vs 0.90). DINOv2 alleine mit Pseudo-Labels ist billiger und mindestens gleich stark.

**V8a = V5-Rezept + Pseudo-Labels:**
- V3-Hyperparams: LR=1e-4, CE + class weights, label_smooth=0.05
- EMA Weights (0.9995), MixUp + CutMix, Heavy Augs
- Layer-wise LR Decay 0.75, Strict CLO (3 Test-Cam Folds)
- **NEU: USE_PSEUDO_LABELS=True**, PSEUDO_CSV=pseudo_labels_t2_v7.csv

## Configuration

In [1]:
TAG = "T2"

import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0,1,2,3"

_candidates = [
    "multiview_pig_posture_recognition",
    "./multiview_pig_posture_recognition",
    "/datasets/multi-view-pig-posture-recognition",
    "/multi-view-pig-posture-recognition",
]
DATA_ROOT = None
for _p in _candidates:
    if os.path.isdir(_p):
        DATA_ROOT = _p
        break
assert DATA_ROOT is not None, "Datenverzeichnis nicht gefunden!"
print(f"DATA_ROOT = {os.path.abspath(DATA_ROOT)}")

if TAG == "T1":
    CSV_PATH = f"{DATA_ROOT}/train1.csv"
    IMG_DIR  = f"{DATA_ROOT}/train1_images"
else:
    CSV_PATH = f"{DATA_ROOT}/train2.csv"
    IMG_DIR  = f"{DATA_ROOT}/train2_images"

OUTPUT_DIR = f"runs/v8a_{TAG.lower()}"

# --- Nur DINOv2 Large @ 518 ---
ARCH_LIST = [
    {
        "name": "vit_large_patch14_dinov2.lvd142m",
        "img_size": 518,
        "batch_size": 16,
        "grad_accum": 2,
        "lr": 1e-4,
        "lr_backbone_mult": 0.1,
        "layer_decay": 0.75,
        "epochs": 30,
        "patience": 10,
        "prefix": "dinov2l",
    },
]

WARMUP_EPOCHS     = 3
LABEL_SMOOTH      = 0.05
PAD_RATIO         = 0.1
NUM_WORKERS       = 16
SEED              = 42
NUM_CLASSES       = 5

MIXUP_ALPHA       = 0.2
CUTMIX_ALPHA      = 1.0
MIX_PROB          = 0.5
SWITCH_PROB       = 0.5

USE_EMA           = True
EMA_DECAY         = 0.9995

CLASS_NAMES = ["Lateral_lying_left", "Lateral_lying_right",
               "Sitting", "Standing", "Sternal_lying"]

LOSS_TYPE    = "ce"

TEST_CAMERAS = ["pen1_tur_cam1", "pen2_orb_cam2", "pen2_tur_cam2"]
VALIDATION_STRATEGY = "test_only"

# --- Pseudo-Labels aus V7 B_dinov2_strong (Kaggle-Best 0.862) ---
USE_PSEUDO_LABELS = True
PSEUDO_CSV        = "pseudo_labels_t2_v7.csv"

PRETRAINED_CKPT   = None

print(f"Tag: {TAG}  |  Architekturen: {len(ARCH_LIST)}")
for a in ARCH_LIST:
    print(f"  - {a['prefix']}: {a['name']} @ {a['img_size']}px  |  BS={a['batch_size']}*{a['grad_accum']}  LR={a['lr']}")
print(f"Pseudo: {USE_PSEUDO_LABELS} ({PSEUDO_CSV})")
print(f"Validation: {VALIDATION_STRATEGY}  |  Loss: {LOSS_TYPE}  |  EMA: {USE_EMA}  |  Output: {OUTPUT_DIR}")

DATA_ROOT = /datasets/multi-view-pig-posture-recognition
Tag: T2  |  Architekturen: 1
  - dinov2l: vit_large_patch14_dinov2.lvd142m @ 518px  |  BS=16*2  LR=0.0001
Pseudo: True (pseudo_labels_t2_v7.csv)
Validation: test_only  |  Loss: ce  |  EMA: True  |  Output: runs/v8a_t2


## Imports

In [2]:
import os, ast, random, re, copy
import numpy as np
import pandas as pd
from PIL import Image
from io import BytesIO
from tqdm.notebook import tqdm
from collections import Counter

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.optim.lr_scheduler import LinearLR, CosineAnnealingLR, SequentialLR
from torch.utils.data import Dataset, DataLoader
from torch.cuda.amp import GradScaler, autocast
import torchvision.transforms as T
import torchvision.transforms.functional as TFn
import timm
from timm.data import Mixup

from sklearn.metrics import f1_score

import warnings
warnings.filterwarnings("ignore")

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {DEVICE}")
if torch.cuda.is_available():
    for i in range(torch.cuda.device_count()):
        name = torch.cuda.get_device_name(i)
        vram = torch.cuda.get_device_properties(i).total_memory / 1e9
        print(f"  GPU {i}: {name} ({vram:.1f} GB)")

def set_seed(seed):
    random.seed(seed); np.random.seed(seed)
    torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)

set_seed(SEED)
os.makedirs(OUTPUT_DIR, exist_ok=True)

<jemalloc>: Unsupported system page size


Device: cuda
  GPU 0: Tesla V100-SXM2-32GB (33.8 GB)
  GPU 1: Tesla V100-SXM2-32GB (33.8 GB)
  GPU 2: Tesla V100-SXM2-32GB (33.8 GB)
  GPU 3: Tesla V100-SXM2-32GB (33.8 GB)


## Daten laden (inkl. Pseudo-Labels)

In [3]:
df = pd.read_csv(CSV_PATH)

def extract_camera(image_id):
    m = re.match(r"(pen\d+_\w+_cam\d+)", image_id)
    return m.group(1) if m else "unknown"

df["camera"] = df["image_id"].apply(extract_camera)
df["img_dir"] = IMG_DIR
df["is_pseudo"] = False

if USE_PSEUDO_LABELS and PSEUDO_CSV and os.path.exists(PSEUDO_CSV):
    pseudo_df = pd.read_csv(PSEUDO_CSV)
    keep_cols = [c for c in pseudo_df.columns if c in ["row_id","image_id","width","height","bbox","class_id"]]
    pseudo_df = pseudo_df[keep_cols]
    pseudo_df["camera"] = pseudo_df["image_id"].apply(extract_camera)
    pseudo_df["img_dir"] = os.path.join(DATA_ROOT, "test_images")
    pseudo_df["is_pseudo"] = True
    df = pd.concat([df, pseudo_df], ignore_index=True)
    print(f"Pseudo-Labels: {len(pseudo_df)} hinzugefuegt")
    print(f"  davon nach Klasse:")
    for c in range(NUM_CLASSES):
        cnt = ((pseudo_df["class_id"]==c)).sum()
        print(f"    {c} - {CLASS_NAMES[c]:<22} {cnt:>5}")
else:
    if USE_PSEUDO_LABELS:
        print(f"WARN: PSEUDO_CSV={PSEUDO_CSV} nicht gefunden -> kein Pseudo-Boost!")

print(f"\nInstanzen total: {len(df)}  |  echt: {(~df['is_pseudo']).sum()}  |  pseudo: {df['is_pseudo'].sum()}")
print(f"Kameras ({df['camera'].nunique()}):")
for cam in sorted(df["camera"].unique()):
    cnt = (df["camera"] == cam).sum()
    is_test = cam in TEST_CAMERAS
    tag = " (TEST)" if is_test else ""
    print(f"  {cam:<25} {cnt:>5} ({100*cnt/len(df):.1f}%){tag}")
print(f"Klassen (inkl. Pseudo):")
for c in range(NUM_CLASSES):
    cnt = (df["class_id"] == c).sum()
    print(f"  {c} - {CLASS_NAMES[c]:<22} {cnt:>5} ({100*cnt/len(df):.1f}%)")

Pseudo-Labels: 1000 hinzugefuegt
  davon nach Klasse:
    0 - Lateral_lying_left       200
    1 - Lateral_lying_right      200
    2 - Sitting                  200
    3 - Standing                 200
    4 - Sternal_lying            200

Instanzen total: 24450  |  echt: 23450  |  pseudo: 1000
Kameras (8):
  pen1_orb_cam1               722 (3.0%)
  pen1_orb_cam2              1488 (6.1%)
  pen1_tur_cam1               631 (2.6%) (TEST)
  pen1_tur_cam2              8194 (33.5%)
  pen2_orb_cam1              2817 (11.5%)
  pen2_orb_cam2               424 (1.7%) (TEST)
  pen2_tur_cam1              9713 (39.7%)
  pen2_tur_cam2               461 (1.9%) (TEST)
Klassen (inkl. Pseudo):
  0 - Lateral_lying_left      3283 (13.4%)
  1 - Lateral_lying_right     3635 (14.9%)
  2 - Sitting                  895 (3.7%)
  3 - Standing               10128 (41.4%)
  4 - Sternal_lying           6509 (26.6%)


## Dataset mit Label-aware Flip

In [4]:
class PigPostureDataset(Dataset):
    def __init__(self, df, transform=None, pad_ratio=0.25, is_train=False, hflip_prob=0.5):
        self.df = df.reset_index(drop=True)
        self.transform = transform
        self.pad_ratio = pad_ratio
        self.is_train = is_train
        self.hflip_prob = hflip_prob

    def __len__(self):
        return len(self.df)

    def _crop(self, img, bbox):
        W, H = img.size
        x, y, w, h = [float(v) for v in ast.literal_eval(bbox)]
        px, py = w * self.pad_ratio, h * self.pad_ratio
        x1 = max(0, int(x - px));  y1 = max(0, int(y - py))
        x2 = min(W, int(x+w+px));  y2 = min(H, int(y+h+py))
        return img.crop((x1, y1, x2, y2))

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img = Image.open(os.path.join(row["img_dir"], row["image_id"])).convert("RGB")
        crop = self._crop(img, row["bbox"])
        label = int(row["class_id"])

        if self.is_train and random.random() < self.hflip_prob:
            crop = TFn.hflip(crop)
            if label == 0: label = 1
            elif label == 1: label = 0

        if self.transform:
            crop = self.transform(crop)
        return crop, label

## Heavy Augmentations (V5-identisch)

In [5]:
class CameraSimTransform:
    def __init__(self, p=0.4):
        self.p = p

    def __call__(self, img):
        if random.random() > self.p:
            return img
        if random.random() < 0.4:
            w, h = img.size
            scale = random.uniform(0.3, 0.7)
            small = img.resize((max(16, int(w*scale)), max(16, int(h*scale))), Image.BILINEAR)
            img = small.resize((w, h), Image.BILINEAR)
        if random.random() < 0.2:
            quality = random.randint(25, 65)
            buffer = BytesIO()
            img.save(buffer, format="JPEG", quality=quality)
            buffer.seek(0)
            img = Image.open(buffer).convert("RGB")
        if random.random() < 0.15:
            arr = np.array(img, dtype=np.float32)
            noise = np.random.normal(0, random.uniform(5, 15), arr.shape)
            arr = np.clip(arr + noise, 0, 255).astype(np.uint8)
            img = Image.fromarray(arr)
        return img


class PerspectiveJitter:
    def __init__(self, distortion_scale=0.15, p=0.4):
        self.distortion_scale = distortion_scale
        self.p = p

    def __call__(self, img):
        if random.random() < self.p:
            d = self.distortion_scale
            w, h = img.size
            return TFn.perspective(
                img,
                startpoints=[[0,0],[w,0],[w,h],[0,h]],
                endpoints=[
                    [int(random.uniform(0, w*d)), int(random.uniform(0, h*d))],
                    [int(w - random.uniform(0, w*d)), int(random.uniform(0, h*d))],
                    [int(w - random.uniform(0, w*d)), int(h - random.uniform(0, h*d))],
                    [int(random.uniform(0, w*d)), int(h - random.uniform(0, h*d))],
                ],
                fill=0
            )
        return img


def get_train_transform(size):
    return T.Compose([
        CameraSimTransform(p=0.4),
        PerspectiveJitter(distortion_scale=0.15, p=0.4),
        T.Resize((int(size*1.15), int(size*1.15)), interpolation=T.InterpolationMode.BICUBIC),
        T.RandomResizedCrop(size, scale=(0.7, 1.0), ratio=(0.85, 1.15),
                            interpolation=T.InterpolationMode.BICUBIC),
        T.RandomRotation(degrees=12),
        T.RandomAffine(degrees=0, scale=(0.85, 1.15), translate=(0.05, 0.05)),
        T.ColorJitter(brightness=0.3, contrast=0.3, saturation=0.2, hue=0.05),
        T.RandomGrayscale(p=0.08),
        T.RandomApply([T.GaussianBlur(kernel_size=5, sigma=(0.1, 2.0))], p=0.2),
        T.ToTensor(),
        T.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
        T.RandomErasing(p=0.25, scale=(0.02, 0.15)),
    ])


def get_val_transform(size):
    return T.Compose([
        T.Resize((size, size), interpolation=T.InterpolationMode.BICUBIC),
        T.ToTensor(),
        T.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
    ])

print("Heavy Augmentations definiert.")

Heavy Augmentations definiert.


## EMA (Exponential Moving Average)

In [6]:
class ModelEMA:
    def __init__(self, model, decay=0.9995, cpu=False):
        self.decay = decay
        self.cpu = cpu
        inner = model.module if isinstance(model, nn.DataParallel) else model
        self.ema = copy.deepcopy(inner).eval()
        for p in self.ema.parameters():
            p.requires_grad_(False)
        if cpu:
            self.ema = self.ema.cpu()

    @torch.no_grad()
    def update(self, model):
        inner = model.module if isinstance(model, nn.DataParallel) else model
        msd = inner.state_dict()
        for k, v in self.ema.state_dict().items():
            if v.dtype.is_floating_point:
                src = msd[k].detach()
                if self.cpu:
                    src = src.cpu()
                v.mul_(self.decay).add_(src, alpha=1 - self.decay)
            else:
                v.copy_(msd[k])

    def state_dict(self):
        return self.ema.state_dict()

## Layer-wise LR Decay (ViT)

In [7]:
def get_vit_layer_id(name, num_layers):
    name = name.replace("module.", "")
    if name.startswith(("cls_token", "pos_embed", "patch_embed", "mask_token", "reg_token")):
        return 0
    if name.startswith("blocks."):
        return int(name.split(".")[1]) + 1
    return num_layers


def build_vit_param_groups(model, base_lr, layer_decay=0.75, weight_decay=1e-2):
    inner = model.module if isinstance(model, nn.DataParallel) else model
    num_layers = len(inner.blocks) if hasattr(inner, "blocks") else 12
    lr_scales = [layer_decay ** (num_layers - i) for i in range(num_layers + 1)]

    groups = {}
    for name, param in inner.named_parameters():
        if not param.requires_grad:
            continue
        lid = get_vit_layer_id(name, num_layers)
        no_wd = param.ndim <= 1 or name.endswith(".bias") or "norm" in name or "cls_token" in name or "pos_embed" in name
        key = (lid, no_wd)
        if key not in groups:
            groups[key] = {
                "params": [],
                "lr": base_lr * lr_scales[lid],
                "weight_decay": 0.0 if no_wd else weight_decay,
                "lr_scale": lr_scales[lid],
            }
        groups[key]["params"].append(param)
    return list(groups.values()), num_layers


def build_optimizer(model, arch_cfg):
    base_lr = arch_cfg["lr"]
    groups, n_layers = build_vit_param_groups(model, base_lr, arch_cfg["layer_decay"])
    print(f"  ViT LLRD: {n_layers} Layers, decay={arch_cfg['layer_decay']}, {len(groups)} groups")
    return optim.AdamW(groups, lr=base_lr)

## Loss: CE + Class Weights

In [8]:
def build_criterion(class_weights, label_smooth=0.05):
    return nn.CrossEntropyLoss(weight=class_weights, label_smoothing=label_smooth)


def soft_ce(logits, targets_soft):
    log_probs = F.log_softmax(logits, dim=-1)
    return -(targets_soft * log_probs).sum(dim=-1).mean()

## Training Helpers

In [9]:
def train_one_epoch(model, loader, optimizer, scaler, criterion,
                    mixup_fn=None, ema=None, grad_accum=1, class_weights=None):
    model.train()
    optimizer.zero_grad()
    loss_sum, n = 0.0, 0
    preds, labels_all = [], []

    for step, (imgs, labels) in enumerate(tqdm(loader, desc="  Train", leave=False)):
        imgs, labels = imgs.to(DEVICE, non_blocking=True), labels.to(DEVICE, non_blocking=True)

        if mixup_fn is not None:
            imgs_m, targets_m = mixup_fn(imgs, labels)
            with autocast():
                logits = model(imgs_m)
                if class_weights is not None:
                    log_probs = F.log_softmax(logits, dim=-1)
                    w = class_weights.unsqueeze(0)
                    loss = -(targets_m * log_probs * w).sum(dim=-1).mean()
                else:
                    loss = soft_ce(logits, targets_m)
                loss = loss / grad_accum
        else:
            with autocast():
                logits = model(imgs)
                loss = criterion(logits, labels) / grad_accum

        scaler.scale(loss).backward()

        if (step + 1) % grad_accum == 0 or (step + 1) == len(loader):
            scaler.unscale_(optimizer)
            nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            scaler.step(optimizer)
            scaler.update()
            optimizer.zero_grad()
            if ema is not None:
                ema.update(model)

        loss_sum += loss.item() * imgs.size(0) * grad_accum
        n += imgs.size(0)
        preds.extend(logits.argmax(1).detach().cpu().numpy())
        labels_all.extend(labels.detach().cpu().numpy())

    return loss_sum / max(n, 1), f1_score(labels_all, preds, average="macro", zero_division=0)


@torch.no_grad()
def validate_epoch(model_or_ema, loader, criterion, ema_on_cpu=False):
    model_or_ema.eval()
    loss_sum, n = 0.0, 0
    preds, labels_all = [], []
    target_device = torch.device("cpu") if ema_on_cpu else DEVICE
    for imgs, labels in tqdm(loader, desc="  Val  ", leave=False):
        imgs  = imgs.to(target_device, non_blocking=True)
        labels = labels.to(target_device, non_blocking=True)
        with autocast(enabled=not ema_on_cpu):
            logits = model_or_ema(imgs)
            loss = criterion(logits, labels)
        loss_sum += loss.item() * imgs.size(0)
        n += imgs.size(0)
        preds.extend(logits.argmax(1).cpu().numpy())
        labels_all.extend(labels.cpu().numpy())
    return (loss_sum / max(n, 1),
            f1_score(labels_all, preds, average="macro", zero_division=0),
            preds, labels_all)

## Strict CLO Folds (Pseudo-Labels nur im Train)

Wichtig: Pseudo-Labels stammen aus test_images und haben test-Kamera-Namen. Sie gehoeren
IMMER ins Train-Set (egal welcher Fold), NIE ins Val-Set - sonst validierst du auf deinen
eigenen Teacher-Predictions.

In [10]:
df_train = df.copy()

available_cams = set(df_train[~df_train["is_pseudo"]]["camera"].unique())
test_cams_in_data = sorted([c for c in TEST_CAMERAS if c in available_cams])

if len(test_cams_in_data) == 0:
    print(f"T1-Modus: keine Test-Kameras -> CLO auf alle Kameras")
    cams_for_clo = sorted(available_cams)
else:
    cams_for_clo = list(test_cams_in_data)
    print(f"STRICT CLO auf Test-Kameras: {cams_for_clo}")

splits = []
fold_cameras = []
for cam in cams_for_clo:
    # Val: echte Samples dieser Kamera (keine Pseudo!)
    val_mask = (df_train["camera"] == cam) & (~df_train["is_pseudo"])
    val_idx = df_train.index[val_mask].values
    # Train: alles andere (inkl. Pseudo-Labels)
    train_idx = df_train.index[~val_mask].values
    if len(val_idx) > 0:
        splits.append((train_idx, val_idx))
        fold_cameras.append(cam)

n_folds = len(splits)
print(f"\n{n_folds} Folds:")
for i, (tr, vl) in enumerate(splits):
    n_pseudo_train = df_train.iloc[tr]["is_pseudo"].sum()
    n_real_train = len(tr) - n_pseudo_train
    print(f"  Fold {i+1}: Val={fold_cameras[i]} ({len(vl)} echt) | "
          f"Train={len(tr)} ({n_real_train} echt + {n_pseudo_train} pseudo)")

STRICT CLO auf Test-Kameras: ['pen1_tur_cam1', 'pen2_orb_cam2', 'pen2_tur_cam2']

3 Folds:
  Fold 1: Val=pen1_tur_cam1 (200 echt) | Train=24250 (23250 echt + 1000 pseudo)
  Fold 2: Val=pen2_orb_cam2 (120 echt) | Train=24330 (23330 echt + 1000 pseudo)
  Fold 3: Val=pen2_tur_cam2 (196 echt) | Train=24254 (23254 echt + 1000 pseudo)


## Training Loop (V8a)

In [ ]:
all_results = {}

for arch_idx, arch in enumerate(ARCH_LIST):
    print(f"\n{'#'*60}")
    print(f"  ARCHITEKTUR {arch_idx+1}/{len(ARCH_LIST)}: {arch['prefix']} ({arch['name']})")
    print(f"{'#'*60}")

    IMG_SIZE_ARCH  = arch["img_size"]
    BATCH_SIZE     = arch["batch_size"]
    GRAD_ACCUM     = arch.get("grad_accum", 1)
    LR             = arch["lr"]
    EPOCHS_ARCH    = arch["epochs"]
    PATIENCE       = arch["patience"]
    PREFIX         = arch["prefix"]

    arch_results = []

    for fold_idx, (train_idx, val_idx) in enumerate(splits):
        print(f"\n{'='*60}")
        print(f"  [{PREFIX}] FOLD {fold_idx + 1} / {n_folds}  (Val={fold_cameras[fold_idx]})")
        print(f"{'='*60}")

        fold_train = df_train.iloc[train_idx].reset_index(drop=True)
        fold_val   = df_train.iloc[val_idx].reset_index(drop=True)

        print(f"  Train: {len(fold_train)} ({fold_train['is_pseudo'].sum()} pseudo) | "
              f"Val: {len(fold_val)} | effBatch={BATCH_SIZE*GRAD_ACCUM}")

        ckpt_path = os.path.join(OUTPUT_DIR, f"best_{PREFIX}_fold_{fold_idx+1}.pth")
        if os.path.exists(ckpt_path):
            ckpt = torch.load(ckpt_path, map_location="cpu")
            print(f"  Checkpoint existiert (val_f1={ckpt.get('val_f1',0):.4f}), skip")
            arch_results.append(ckpt.get("val_f1", 0))
            continue

        train_ds = PigPostureDataset(fold_train, transform=get_train_transform(IMG_SIZE_ARCH),
                                     pad_ratio=PAD_RATIO, is_train=True, hflip_prob=0.5)
        val_ds   = PigPostureDataset(fold_val, transform=get_val_transform(IMG_SIZE_ARCH),
                                     pad_ratio=PAD_RATIO, is_train=False)
        train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,
                                  num_workers=NUM_WORKERS, pin_memory=True, drop_last=True,
                                  persistent_workers=(NUM_WORKERS > 0))
        val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False,
                                  num_workers=NUM_WORKERS, pin_memory=True,
                                  persistent_workers=(NUM_WORKERS > 0))

        try:
            model = timm.create_model(arch["name"], pretrained=True,
                                      num_classes=NUM_CLASSES, img_size=IMG_SIZE_ARCH)
        except TypeError:
            model = timm.create_model(arch["name"], pretrained=True, num_classes=NUM_CLASSES)

        if PRETRAINED_CKPT and os.path.exists(PRETRAINED_CKPT):
            ck = torch.load(PRETRAINED_CKPT, map_location="cpu")
            model.load_state_dict(ck["model"], strict=False)

        model = model.to(DEVICE)
        if torch.cuda.device_count() > 1:
            model = nn.DataParallel(model)

        params = sum(p.numel() for p in model.parameters()) / 1e6
        print(f"  Modell: {arch['name']} ({params:.1f}M)")

        ema = ModelEMA(model, decay=EMA_DECAY, cpu=False) if USE_EMA else None
        if ema is not None:
            print(f"  EMA aktiv (decay={EMA_DECAY})")

        counts = Counter(fold_train["class_id"].tolist())
        class_weights = torch.tensor(
            [len(fold_train) / (NUM_CLASSES * max(counts.get(c, 1), 1))
             for c in range(NUM_CLASSES)], dtype=torch.float32
        ).to(DEVICE)
        criterion = build_criterion(class_weights, label_smooth=LABEL_SMOOTH)
        print(f"  Loss: CE | Klassen-Gewichte: {[f'{w:.2f}' for w in class_weights.cpu().tolist()]}")

        mixup_fn = Mixup(
            mixup_alpha=MIXUP_ALPHA, cutmix_alpha=CUTMIX_ALPHA,
            prob=MIX_PROB, switch_prob=SWITCH_PROB,
            label_smoothing=LABEL_SMOOTH, num_classes=NUM_CLASSES,
        )
        print(f"  MixUp a={MIXUP_ALPHA} / CutMix a={CUTMIX_ALPHA} / prob={MIX_PROB}")

        optimizer = build_optimizer(model, arch)
        warmup = LinearLR(optimizer, start_factor=0.01, total_iters=WARMUP_EPOCHS)
        cosine = CosineAnnealingLR(optimizer, T_max=EPOCHS_ARCH - WARMUP_EPOCHS, eta_min=1e-7)
        scheduler = SequentialLR(optimizer, schedulers=[warmup, cosine], milestones=[WARMUP_EPOCHS])
        scaler = GradScaler()

        best_val_f1 = 0.0
        patience_counter = 0
        best_source = "model"

        for epoch in range(1, EPOCHS_ARCH + 1):
            train_loss, train_f1 = train_one_epoch(
                model, train_loader, optimizer, scaler, criterion,
                mixup_fn=mixup_fn, ema=ema, grad_accum=GRAD_ACCUM, class_weights=class_weights
            )
            val_loss, val_f1, val_preds, val_labels = validate_epoch(model, val_loader, criterion)
            source = "model"
            if ema is not None:
                e_loss, e_f1, e_preds, e_labels = validate_epoch(ema.ema, val_loader, criterion)
                if e_f1 > val_f1:
                    val_loss, val_f1, val_preds, val_labels = e_loss, e_f1, e_preds, e_labels
                    source = "ema"
            scheduler.step()

            improved = val_f1 > best_val_f1
            mark = "*" if improved else " "
            phase = "warmup" if epoch <= WARMUP_EPOCHS else "cosine"
            print(f"  {mark} Epoch {epoch:02d}/{EPOCHS_ARCH} [{phase}] | "
                  f"Train L={train_loss:.4f} F1={train_f1:.4f} | "
                  f"Val[{source}] L={val_loss:.4f} F1={val_f1:.4f}"
                  f"{' <- BEST' if improved else ''}")

            if improved:
                best_val_f1 = val_f1
                best_source = source
                patience_counter = 0
                if source == "ema" and ema is not None:
                    state = ema.state_dict()
                else:
                    state = (model.module.state_dict() if isinstance(model, nn.DataParallel)
                             else model.state_dict())
                torch.save({
                    "epoch": epoch, "model": state,
                    "val_f1": val_f1, "model_name": arch["name"], "arch_prefix": PREFIX,
                    "tag": TAG, "img_size": IMG_SIZE_ARCH, "pad_ratio": PAD_RATIO,
                    "val_camera": fold_cameras[fold_idx], "source": source,
                }, ckpt_path)
            else:
                patience_counter += 1
                if patience_counter >= PATIENCE:
                    print(f"  Early Stopping nach {PATIENCE} Epochen")
                    break

        print(f"\n  Klassifikation Fold {fold_idx+1} (source={best_source}):")
        for c in range(NUM_CLASSES):
            mask = np.array(val_labels) == c
            if mask.sum() > 0:
                correct = (np.array(val_preds)[mask] == c).sum()
                print(f"    {CLASS_NAMES[c]:<22} {correct}/{mask.sum()} ({100*correct/max(mask.sum(),1):.0f}%)")

        arch_results.append(best_val_f1)
        print(f"  Best Val F1: {best_val_f1:.4f}  ({best_source})")

        try:
            del train_loader, val_loader, train_ds, val_ds
        except NameError:
            pass
        del model, optimizer, scheduler, scaler
        if ema is not None:
            del ema
        import gc
        gc.collect()
        torch.cuda.synchronize() if torch.cuda.is_available() else None
        torch.cuda.empty_cache()
        torch.cuda.ipc_collect() if torch.cuda.is_available() else None

    all_results[PREFIX] = arch_results
    mean_f1 = float(np.mean(arch_results))
    print(f"\n  [{PREFIX}] Durchschnitt ueber {len(arch_results)} Folds: {mean_f1:.4f}")

print(f"\n{'#'*60}")
print(f"  GESAMT-ERGEBNIS V8a")
print(f"{'#'*60}")
for prefix, results in all_results.items():
    print(f"  {prefix:<12} | Mean F1: {np.mean(results):.4f} (+/- {np.std(results):.4f}) | Folds: {results}")


############################################################
  ARCHITEKTUR 1/1: dinov2l (vit_large_patch14_dinov2.lvd142m)
############################################################

  [dinov2l] FOLD 1 / 3  (Val=pen1_tur_cam1)
  Train: 24250 (1000 pseudo) | Val: 200 | effBatch=32
  Checkpoint existiert (val_f1=0.8633), skip

  [dinov2l] FOLD 2 / 3  (Val=pen2_orb_cam2)
  Train: 24330 (1000 pseudo) | Val: 120 | effBatch=32


Xet Storage is enabled for this repo, but the 'hf_xet' package is not installed. Falling back to regular HTTP download. For better performance, install the package with: `pip install huggingface_hub[hf_xet]` or `pip install hf_xet`


model.safetensors:   0%|          | 0.00/1.22G [00:00<?, ?B/s]

  Modell: vit_large_patch14_dinov2.lvd142m (304.4M)
  EMA aktiv (decay=0.9995)
  Loss: CE | Klassen-Gewichte: ['1.49', '1.35', '5.44', '0.48', '0.75']
  MixUp a=0.2 / CutMix a=1.0 / prob=0.5
  ViT LLRD: 24 Layers, decay=0.75, 50 groups


  Train:   0%|          | 0/1520 [00:00<?, ?it/s]

...

  Train:   0%|          | 0/1520 [00:00<?, ?it/s]

  Val  :   0%|          | 0/8 [00:00<?, ?it/s]

  Val  :   0%|          | 0/8 [00:00<?, ?it/s]

  * Epoch 06/30 [cosine] | Train L=0.6757 F1=0.7867 | Val[ema] L=0.5477 F1=0.9473 <- BEST


  Train:   0%|          | 0/1520 [00:00<?, ?it/s]